# Figure 2 — Language localizer bar charts

4-panel bar chart figure:
- **Panel 1** English vs French (AliceEn vs AliceFr)
- **Panel 2** L1 vs L2 (dominance-based)
- **Panel 3** Session effect
- **Panel 4** Modality (auditory vs visual)

Each dot = one subject × one session. Bars = group mean ± SEM.

**Inputs** (from env vars injected by `invoke run-notebooks`):
- `OUTPUT_DATA_DIR` — pipeline output root


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from nilearn.image import resample_to_img

OUTPUT_DIR  = Path(os.environ.get('OUTPUT_DATA_DIR', 'output_data'))
LANGLOC_DIR = OUTPUT_DIR / 'langlocalizer'
FROI_DIR    = LANGLOC_DIR / 'fedorenko_frois'

FIGURES_OUTPUT_DIR = Path(os.environ.get('FIGURES_OUTPUT_DIR',
                          str(OUTPUT_DIR / 'figures_manuscript')))
FIG_DIR = FIGURES_OUTPUT_DIR / 'fig2_langloc_barcharts'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output dir : {OUTPUT_DIR}')
print(f'fROI dir   : {FROI_DIR}')
print(f'Figure dir : {FIG_DIR}')

Output dir : /scratch/ibilgin/cneuromod.glm/output_data
fROI dir   : /scratch/ibilgin/cneuromod.glm/output_data/langlocalizer/fedorenko_frois
Figure dir : /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig2_langloc_barcharts


In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
TOP_PERCENT = 0.10  # must match froi_top_percent in invoke.yaml

# Task metadata: contrast, modality, language label
TASK_META = {
    'aliceEn':   {'contrast': 'int-degr', 'modality': 'auditory', 'language': 'English'},
    'aliceFr':   {'contrast': 'int-degr', 'modality': 'auditory', 'language': 'French'},
    'listening': {'contrast': 'int-degr', 'modality': 'auditory', 'language': None},
    'reading':   {'contrast': 'word-nonword', 'modality': 'visual', 'language': None},
}

# Subject language dominance
LANGUAGE_DOMINANCE = {
    'sub-01': {'L1': 'French',  'L2': 'English'},
    'sub-02': {'L1': 'French',  'L2': 'English'},
    'sub-03': {'L1': 'English', 'L2': 'French'},
    'sub-05': {'L1': 'English', 'L2': 'French'},
}

SUBJECT_COLORS = {
    'sub-01': '#E63946',
    'sub-02': '#06A77D',
    'sub-03': '#457B9D',
    'sub-05': '#F77F00',
    'sub-06': '#9D4EDD',
}
BAR_COLORS = {
    'language': '#B8D4E8',
    'session':  '#E8D4C8',
    'modality': '#D4E8D8',
}

In [3]:
# ---------------------------------------------------------------------------
# Data extraction
# ---------------------------------------------------------------------------

def load_froi_mask(subject, session, task, roi_type='allLANG'):
    """
    Return a union fROI mask image for subject/session/task.
    roi_type: 'allLANG' | 'LH' | 'RH' | parcel name
    """
    contrast = TASK_META[task]['contrast']

    if roi_type == 'allLANG':
        pattern = f'{subject}_{session}_task-{task}_contrast-{contrast}_parcel-*_top.nii.gz'
        files = sorted(FROI_DIR.glob(pattern))
    elif roi_type in ('LH', 'RH'):
        pattern = f'{subject}_{session}_task-{task}_contrast-{contrast}_parcel-{roi_type}_*_top.nii.gz'
        files = sorted(FROI_DIR.glob(pattern))
    else:
        pattern = f'{subject}_{session}_task-{task}_contrast-{contrast}_parcel-{roi_type}_top.nii.gz'
        files = sorted(FROI_DIR.glob(pattern))

    if not files:
        return None

    ref   = nib.load(str(files[0]))
    union = np.zeros(ref.shape, dtype=bool)
    for f in files:
        union |= (nib.load(str(f)).get_fdata() > 0)
    return nib.Nifti1Image(union.astype(np.uint8), ref.affine, ref.header)


def extract_froi_data(roi_type='allLANG'):
    """
    Extract mean z-score within fROI for all subjects / sessions / tasks.
    Returns a DataFrame with columns:
      subject, session, task, contrast, modality, language, value, roi_type
    """
    rows = []
    subjects = sorted(set(
        f.name.split('_')[0]
        for f in FROI_DIR.glob('sub-*_ses-*_task-*_parcel-*_top.nii.gz')
    ))

    for subject in subjects:
        subj_glm = LANGLOC_DIR / subject
        if not subj_glm.exists():
            continue

        for task, meta in TASK_META.items():
            contrast = meta['contrast']
            for ses_dir in sorted(subj_glm.glob('ses-*')):
                session = ses_dir.name
                z_path  = ses_dir / task / f'{subject}_{session}_task-{task}_contrast-{contrast}_stat-z.nii.gz'
                if not z_path.exists():
                    continue

                mask_img = load_froi_mask(subject, session, task, roi_type)
                if mask_img is None:
                    continue

                z_img  = nib.load(str(z_path))
                z_data = z_img.get_fdata()
                mask   = mask_img.get_fdata() > 0

                if mask.shape != z_data.shape:
                    mask_img = resample_to_img(mask_img, z_img, interpolation='nearest')
                    mask     = mask_img.get_fdata() > 0

                if mask.sum() == 0:
                    continue

                rows.append({
                    'subject':  subject,
                    'session':  session,
                    'task':     task,
                    'contrast': contrast,
                    'modality': meta['modality'],
                    'language': meta['language'],
                    'value':    float(np.nanmean(z_data[mask])),
                    'roi_type': roi_type,
                })

    return pd.DataFrame(rows)

In [4]:
# ---------------------------------------------------------------------------
# Plotting helpers
# ---------------------------------------------------------------------------

def _style_ax(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['left'].set_color('#34495E')
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['bottom'].set_color('#34495E')
    ax.axhline(0, color='#7F8C8D', linewidth=1.5, linestyle='--', alpha=0.6, zorder=1)
    ax.tick_params(axis='both', labelsize=12, colors='#34495E', width=1.5, length=6)
    ax.grid(axis='y', alpha=0.25, linestyle='--', linewidth=1, color='#BDC3C7')
    ax.set_ylim([0, 8.5])
    ax.set_yticks(np.arange(0, 9, 1))
    ax.set_ylabel('Mean z-score', fontsize=16, fontweight='600', color='#2C3E50', labelpad=10)


def _draw_bars_and_dots(ax, x_pos, labels, group_data, df, x_col, bar_color, french_edge=False):
    """
    Draw bars + error bars + jittered subject dots.
    group_data: list of (mean, sem) per x position
    df: DataFrame with x_col, subject, value columns
    """
    means = [g[0] for g in group_data]
    sems  = [g[1] for g in group_data]

    ax.bar(x_pos, means, 0.65, color=bar_color, alpha=0.8,
           edgecolor='#2C3E50', linewidth=1.8, zorder=2)
    ax.errorbar(x_pos, means, yerr=sems, fmt='none', color='#2C3E50',
                capsize=7, linewidth=2.5, capthick=2.5, zorder=3)

    for xi, label in zip(x_pos, labels):
        rows = df[df[x_col] == label]
        for _, row in rows.iterrows():
            jitter = np.random.uniform(-0.12, 0.12)
            edge   = 'white' if (french_edge and row.get('language') == 'French') else 'none'
            ax.scatter(xi + jitter, row['value'],
                       color=SUBJECT_COLORS.get(row['subject'], '#7F8C8D'),
                       s=200, zorder=10, marker='o',
                       edgecolor=edge, linewidth=2.5, alpha=0.95)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=14, color='#34495E')

In [5]:
# ---------------------------------------------------------------------------
# Figure builder
# ---------------------------------------------------------------------------

def make_figure2(df, roi_type='allLANG', dpi=300):
    """
    4-panel subject specificity figure.
    """
    if df is None or len(df) == 0:
        print(f'[SKIP] No data for roi_type={roi_type}')
        return None

    np.random.seed(42)  # reproducible jitter

    fig = plt.figure(figsize=(24, 7))
    gs  = GridSpec(1, 4, wspace=0.35, left=0.06, right=0.96, top=0.85, bottom=0.12)

    # -- Panel 1: English vs French --
    ax1   = fig.add_subplot(gs[0])
    df_al = df[df['task'].isin(['aliceEn', 'aliceFr'])].copy()
    df_al['language_label'] = df_al['language']
    langs = ['English', 'French']
    gd    = [(df_al[df_al['language_label'] == l]['value'].mean(),
              df_al[df_al['language_label'] == l]['value'].sem()) for l in langs]
    _draw_bars_and_dots(ax1, np.arange(2), langs, gd, df_al, 'language_label',
                        BAR_COLORS['language'], french_edge=True)
    ax1.set_xticklabels(['English\n(AliceEn)', 'French\n(AliceFr)'], fontsize=14)
    ax1.set_title('English vs French', fontsize=17, fontweight='700', color='#2C3E50', pad=15)
    _style_ax(ax1)

    # -- Panel 2: L1 vs L2 --
    ax2 = fig.add_subplot(gs[1])
    df_dom = df_al.copy()
    def _dominance(row):
        dom = LANGUAGE_DOMINANCE.get(row['subject'])
        if dom is None: return None
        if row['language'] == dom['L1']: return 'L1'
        if row['language'] == dom['L2']: return 'L2'
        return None
    df_dom['dominance'] = df_dom.apply(_dominance, axis=1)
    df_dom = df_dom.dropna(subset=['dominance'])
    doms = ['L1', 'L2']
    gd2  = [(df_dom[df_dom['dominance'] == d]['value'].mean(),
             df_dom[df_dom['dominance'] == d]['value'].sem()) for d in doms]
    _draw_bars_and_dots(ax2, np.arange(2), doms, gd2, df_dom, 'dominance',
                        BAR_COLORS['language'], french_edge=True)
    ax2.set_xticklabels(['L1\n(Dominant)', 'L2\n(Non-dominant)'], fontsize=14)
    ax2.set_title('L1 vs L2', fontsize=17, fontweight='700', color='#2C3E50', pad=15)
    _style_ax(ax2)

    # -- Panel 3: Session effect --
    ax3  = fig.add_subplot(gs[2])
    sess = sorted(df['session'].unique())
    sess_summary = df.groupby(['subject', 'session'])['value'].mean().reset_index()
    gd3  = [(sess_summary[sess_summary['session'] == s]['value'].mean(),
             sess_summary[sess_summary['session'] == s]['value'].sem()) for s in sess]
    _draw_bars_and_dots(ax3, np.arange(len(sess)), sess, gd3, sess_summary, 'session',
                        BAR_COLORS['session'])
    ax3.set_xticklabels([s.replace('ses-00', '') for s in sess], fontsize=14)
    ax3.set_title('Session', fontsize=17, fontweight='700', color='#2C3E50', pad=15)
    _style_ax(ax3)

    # -- Panel 4: Modality --
    ax4  = fig.add_subplot(gs[3])
    mods = ['auditory', 'visual']
    gd4  = [(df[df['modality'] == m]['value'].mean(),
             df[df['modality'] == m]['value'].sem()) for m in mods]
    _draw_bars_and_dots(ax4, np.arange(2), mods, gd4, df, 'modality',
                        BAR_COLORS['modality'], french_edge=True)
    ax4.set_xticklabels(['Auditory', 'Visual'], fontsize=14)
    ax4.set_title('Modality', fontsize=17, fontweight='700', color='#2C3E50', pad=15)
    ax4.set_ylim([0, 9])
    _style_ax(ax4)

    # Super-title
    roi_label = ('Mean of all fROIs' if roi_type == 'allLANG'
                 else 'Left Hemisphere' if roi_type == 'LH'
                 else 'Right Hemisphere' if roi_type == 'RH'
                 else roi_type)
    fig.text(0.5, 0.97, f'Subject Specificity: {roi_label}',
             ha='center', fontsize=20, fontweight='700', color='#2C3E50')
    fig.text(0.5, 0.92,
             'Bars = group mean ± SEM  |  Each dot = one subject in one session',
             ha='center', fontsize=11, style='italic', color='#566573')

    # Legend
    subjects = sorted(df['subject'].unique())
    handles  = [plt.scatter([], [], s=180,
                             color=SUBJECT_COLORS.get(s, '#7F8C8D'),
                             edgecolor='white', linewidth=2.5, alpha=0.95)
                for s in subjects]
    fig.legend(handles, subjects, loc='lower center', ncol=len(subjects),
               frameon=True, fontsize=13, bbox_to_anchor=(0.5, -0.02),
               fancybox=True, facecolor='white', edgecolor='#BDC3C7')

    plt.tight_layout(rect=[0, 0.03, 1, 0.90])

    safe_roi  = roi_type.replace(' ', '_').replace('/', '-')
    out_path  = FIG_DIR / f'fig2_langloc_{safe_roi}.png'
    out_pdf   = FIG_DIR / f'fig2_langloc_{safe_roi}.pdf'
    fig.savefig(str(out_path), dpi=dpi, bbox_inches='tight', facecolor='white')
    fig.savefig(str(out_pdf),  bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  Saved: {out_path.name}')
    return out_path

In [6]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
froi_files = list(FROI_DIR.glob('sub-*_ses-*_task-*_parcel-*_top.nii.gz'))

if not froi_files:
    print('[SKIP] No session-level fROI files found — run `invoke run-froi` first.')
else:
    for roi_type in ['allLANG', 'LH', 'RH']:
        print(f'\nExtracting data for roi_type={roi_type} ...')
        df = extract_froi_data(roi_type=roi_type)
        print(f'  {len(df)} rows extracted')
        if len(df) > 0:
            # Save CSV for downstream use
            csv_path = FIG_DIR / f'data_fig2_{roi_type}.csv'
            df.to_csv(csv_path, index=False)
            make_figure2(df, roi_type=roi_type)

    print(f'\nFigures saved to: {FIG_DIR}')


Extracting data for roi_type=allLANG ...


  48 rows extracted


  Saved: fig2_langloc_allLANG.png

Extracting data for roi_type=LH ...


  48 rows extracted


  Saved: fig2_langloc_LH.png

Extracting data for roi_type=RH ...


  48 rows extracted


  Saved: fig2_langloc_RH.png

Figures saved to: /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig2_langloc_barcharts
